In [1]:
from tmu.models.classification.vanilla_classifier import TMClassifier
import numpy as np
import os 
import logging
import config
import pickle
from utils.optuna import save_optuna_best_results, load_optuna_best_results, run_tm_optuna_search
from utils.dataset import load_dataset
from utils.plots import plot_tm_roc_curve, plot_confusion_matrix_from_scores, plot_clause_length_distribution
from utils.booleanization import perform_color_thermometers_booleanization
from utils.training import train_tm_and_save_scores

/home/coder/masters_thesis/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-03-11 21:43:09,451 - matplotlib - DEBUG - matplotlib data path: /home/coder/masters_thesis/venv/lib/python3.12/site-packages/matplotlib/mpl-data


2026-03-11 21:43:09,456 - matplotlib - DEBUG - CONFIGDIR=/home/coder/.config/matplotlib


2026-03-11 21:43:09,471 - matplotlib - DEBUG - interactive is False


2026-03-11 21:43:09,471 - matplotlib - DEBUG - platform is linux


2026-03-11 21:43:09,525 - matplotlib - DEBUG - CACHEDIR=/home/coder/.cache/matplotlib


2026-03-11 21:43:09,526 - matplotlib.font_manager - DEBUG - Using fontManager instance from /home/coder/.cache/matplotlib/fontlist-v390.json


In [2]:
resolution=4

In [3]:
logging.getLogger("tmu").setLevel(logging.WARNING)
logging.getLogger("tmu.clause_bank.clause_bank_cuda").setLevel(logging.WARNING)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

In [4]:
X_train_org, Y_train, X_val_org, Y_val, X_test_org, Y_test = load_dataset(config)

In [5]:
X_train, X_val, X_test, Y_train, Y_val, Y_test = perform_color_thermometers_booleanization(X_train_org, X_val_org, X_test_org, Y_train, Y_val, Y_test, resolution)

In [6]:
X_train = X_train.astype(np.uint32)
X_val   = X_val.astype(np.uint32)
X_test  = X_test.astype(np.uint32)
Y_test  = Y_test.astype(np.uint32)
Y_train  = Y_train.astype(np.uint32)

In [7]:
study = run_tm_optuna_search(
    TMClassifier,
    config,
    X_train,
    Y_train,
    X_val,
    Y_val,
    metric="f1_macro",
)

[I 2026-03-11 21:43:10,577] A new study created in memory with name: no-name-99079707-a330-4a1b-8623-a014e6e71df4


[I 2026-03-12 02:00:40,756] Trial 0 finished with value: 0.8999900827184697 and parameters: {'s': 6.0, 'T': 10000, 'patch_size': 3, 'max_included_literals': 64}. Best is trial 0 with value: 0.8999900827184697.


[I 2026-03-12 06:33:21,937] Trial 1 finished with value: 0.8868621535251716 and parameters: {'s': 1.0, 'T': 10000, 'patch_size': 3, 'max_included_literals': 64}. Best is trial 0 with value: 0.8999900827184697.


[I 2026-03-12 15:43:30,501] Trial 2 finished with value: 0.9003817899350535 and parameters: {'s': 10.0, 'T': 2000, 'patch_size': 9, 'max_included_literals': 128}. Best is trial 2 with value: 0.9003817899350535.


[I 2026-03-12 23:02:39,686] Trial 3 finished with value: 0.9010717698642013 and parameters: {'s': 10.0, 'T': 2500, 'patch_size': 7, 'max_included_literals': 128}. Best is trial 3 with value: 0.9010717698642013.


...

[I 2026-03-13 15:17:46,003] Trial 7 finished with value: 0.896981656436489 and parameters: {'s': 12.0, 'T': 2500, 'patch_size': 5, 'max_included_literals': 32}. Best is trial 6 with value: 0.9116816626257129.


[I 2026-03-13 20:00:47,316] Trial 8 finished with value: 0.8914663682150658 and parameters: {'s': 13.0, 'T': 6500, 'patch_size': 3, 'max_included_literals': 128}. Best is trial 6 with value: 0.9116816626257129.


[I 2026-03-14 00:39:33,960] Trial 9 finished with value: 0.9159477324784702 and parameters: {'s': 2.0, 'T': 7500, 'patch_size': 7, 'max_included_literals': 32}. Best is trial 9 with value: 0.9159477324784702.


[I 2026-03-14 06:46:27,285] Trial 10 finished with value: 0.8949615398108051 and parameters: {'s': 4.0, 'T': 7500, 'patch_size': 7, 'max_included_literals': 32}. Best is trial 9 with value: 0.9159477324784702.


[I 2026-03-14 12:52:06,171] Trial 11 finished with value: 0.8969471814879957 and parameters: {'s': 6.0, 'T': 4500, 'patch_size': 7, 'max_included_literals': 512}. Best is trial 9 with value: 0.9159477324784702.


In [ ]:
best_params = study.best_params
print("Best F1 score macro accuracy:", study.best_value)
print("Best hyperparameters:", best_params)
save_optuna_best_results(study, "hyperparameters/color_thermometers_resolution_4.json")

In [ ]:
best_value, best_params = load_optuna_best_results("hyperparameters/color_thermometers_resolution_4.json")
s_best = best_params["s"]
T_best = best_params["T"]
patch_best = best_params["patch_size"]
weighted_best = 1
maxlit_best = best_params["max_included_literals"]

In [ ]:
tm = train_tm_and_save_scores(
    TMClassifier=TMClassifier,
    config=config,
    X_train=X_train,
    Y_train=Y_train,
    X_test=X_test,
    Y_test=Y_test,
    T=T_best,
    s=s_best,
    patch_dim=patch_best,
    max_included_literals=maxlit_best,
    weighted_clauses=weighted_best,
    specialist_name="BloodMNIST_Color_Thermometers_resolution_4",
    save_dir=config.SAVE_DIR_SCORES,
)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
model_path = os.path.join("saved_models", "BloodMNIST_Color_Thermometers_Specialist_resolution_4.pkl")

In [ ]:
if not os.path.exists(model_path):
    with open(model_path, "wb") as f:
        pickle.dump(tm, f)
    print("Model saved.")
else:
    print("Model already exists. Skipping save.")

In [ ]:
with open(model_path, "rb") as f:
    tm = pickle.load(f)

In [ ]:
plot_tm_roc_curve(config, T_best, s_best, patch_best, maxlit_best, weighted_best, 
                Y_test, "BloodMNIST_Color_Thermometers_resolution_4")

In [ ]:
_ = plot_clause_length_distribution(tm)

In [ ]:
_, _ = plot_confusion_matrix_from_scores(config, "BloodMNIST_Color_Thermometers_resolution_4", Y_test) 